• Since this is an artificial scenario, you can simply assume that each active pod 
consumes 200W of electricity. As workloads run 40s on average and we have 
180 workloads in total, the expected overall energy usage is40s * 180 * 200W 
= 400 Wh. What is the carbon footprint? 

In [28]:
import pandas as pd
import re

def create_df(log_file):
    # Define regex pattern to extract relevant fields
    log_pattern = re.compile(
        r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) - INFO - Pod: (?P<pod>\S+), "
        r"Node: (?P<node>\S+), Intensity: (?P<intensity>[\d.]+), Region: (?P<region>\S+), Type: (?P<type>\S+)"
    )

    # Read and parse the log file
    data = []
    with open(log_file, "r") as file:
        for line in file:
            match = log_pattern.search(line)
            if match:
                data.append(match.groupdict())

    # Create DataFrame
    df = pd.DataFrame(data)

    # Convert columns to appropriate types
    df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime
    df["intensity"] = df["intensity"].astype(float)  # Convert intensity to float
    
    # Display DataFrame
    print(df.head())
    return df

normal_log_file = "results/normal_strategy_31_01.log"
carbon_log_file = "results/carbonaware_strategy_31_01.log"

normal = create_df(normal_log_file)
carbon = create_df(carbon_log_file)

            timestamp                  pod                  node  intensity  \
0 2025-01-31 15:52:02  workload-1738338722        normal-worker2     233.48   
1 2025-01-31 15:52:03  workload-1738338722        normal-worker2     233.48   
2 2025-01-31 15:52:13  workload-1738338733        normal-worker2     233.48   
3 2025-01-31 15:52:14  workload-1738338733        normal-worker2     233.48   
4 2025-01-31 15:52:22  workload-1738338742  normal-control-plane     195.25   

  region     type  
0     NL  Planned  
1     NL   Actual  
2     NL  Planned  
3     NL   Actual  
4     DE  Planned  
            timestamp                  pod                node  intensity  \
0 2025-01-31 15:52:02  workload-1738338722  carbonaware-worker     156.45   
1 2025-01-31 15:52:02  workload-1738338722  carbonaware-worker     156.45   
2 2025-01-31 15:52:13  workload-1738338733  carbonaware-worker     156.45   
3 2025-01-31 15:52:13  workload-1738338733  carbonaware-worker     156.45   
4 2025-01-31 15:52:2

C:\Users\morit\AppData\Local\Temp\ipykernel_18316\768959533.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime
C:\Users\morit\AppData\Local\Temp\ipykernel_18316\768959533.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime


Normal

In [29]:
def analyze_df(df):
    # Calculate average intensity overall and per node
    avg_intensity_overall = df["intensity"].mean()
    avg_intensity_per_node = df.groupby("node")["intensity"].mean()
    workload_count_per_node = df["node"].filter(like='Actual', axis=0).value_counts(normalize=True) * 100
    avg_footprint = avg_intensity_overall * 0.4

    # Display results
    print("Average Intensity Overall:", avg_intensity_overall)
    print("Average Footprint Overall:", avg_footprint)
    print("Average Intensity Per Node:", avg_intensity_per_node, end= "\n\n")
    print("workload_count_per_node" , workload_count_per_node, end= "\n\n")


In [30]:
analyze_df(normal)

Average Intensity Overall: 284.387321937322
Average Footprint Overall: 113.7549287749288
Average Intensity Per Node: node
normal-control-plane    356.974407
normal-worker           229.100299
normal-worker2          272.702323
Name: intensity, dtype: float64

workload_count_per_node Series([], Name: proportion, dtype: float64)



Carbon

In [31]:
analyze_df(carbon)

Average Intensity Overall: 218.38546478873238
Average Footprint Overall: 87.35418591549296
Average Intensity Per Node: node
carbonaware-worker     209.596345
carbonaware-worker2    257.598462
Name: intensity, dtype: float64

workload_count_per_node Series([], Name: proportion, dtype: float64)

